# Formal Semantic Parsing with Language Models

Some useful functions and some functions that you need to implement are presented in this notebook. You don't have to use them though.

In this assignment, we will evaluate the ability of language on semantic parsing task. In particular, SQL parsing. The assignment has three parts:


1.   Basic Prompt
2.   Finetuning
3.   Context-Free Grammer

In each parts, you will exaime the model output in terms of correctness and output well-formedness.

Part 2 and 3 can be replaced by other means such as RAG. Students are wellcomed to propose their own ideas to solve this task.

References:
https://github.com/jkkummerfeld/text2sql-data/

https://github.com/mlc-ai/xgrammar

**This starting code is based on old transformers-cfg, you have to use xgrammar. You need to check the official document of xgrammar or Lab7. There are some functions for you to fill to**

**You can run small model on Kaggle notebook https://www.kaggle.com/ for free**



In [1]:
!pip install transformers datasets trl peft
!pip install xgrammar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.4/517.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 44.4 MB/s eta 0:00:00


In [2]:
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.added-in-2020.sqlite
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-fields.txt
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-schema.csv
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography.json
!wget https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.sql

--2025-12-18 06:36:39--  https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.added-in-2020.sqlite
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 65536 (64K) [application/octet-stream]
Saving to: ‘geography-db.added-in-2020.sqlite’

geography-db.added- 100%[===================>]  64.00K  --.-KB/s    in 0.001s  

2025-12-18 06:36:39 (47.4 MB/s) - ‘geography-db.added-in-2020.sqlite’ saved [65536/65536]

--2025-12-18 06:36:39--  https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-fields.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443

In [3]:
import json
# The original GeoQuery data with variable placeholders in geography.

# json is expanded into a sample list sorted by train/dev/test, and each sample contains: natural language question, corresponding gold SQL(variable has been replaced), and a splicing string text for training the model.

def extract_sentence_fields(sentence):
    text = sentence["text"]                          # taking out the text of natural language questions
    variables = sentence["variables"]                # { "CITY0": "boston" }
    split = sentence["question-split"]               # finding out which data it belongs to
    return text, variables, split

def insert_variables(sql, sql_variables, sent, sent_variables):
    # SQL _ variables: A list of variables declared in SQL(chich contains info: name, example...)
    # sent: question sentence strings
    # sent_variables: the variable value dict of this sentence


    for info in sql_variables:
        name = info['name']
        value = info['example']
        if name in sent_variables and sent_variables[name] != "":
        # if the variable is provided in the variables of this sentence and it is not an empty string
            value = sent_variables[name]
        sent = value.join(sent.split(name))
        # replace all name in sent with value
        qvalue = '{}'.format(value)
        # take value into a string
        sql = qvalue.join(sql.split(name))
    return (sql, sent)



def build_question_split(jsons,making_prompt=lambda x:x, keep_variables=False):            # expanding the whole json into datasets
    datasets = {}
    for json_dict in jsons:
        for query in [json_dict["sql"][0]]:           # take out the SQL statement from the dictionary
            sql_vars = json_dict['variables']         # get the variable definition list of SQL
            for sentence in json_dict["sentences"]:
                text, variables, split = extract_sentence_fields(sentence)
                if split == "exclude":
                    continue
                if keep_variables:
                    sql = query
                    question = text
                else:
                    sql, question = insert_variables(
                        query, sql_vars, text, variables)       # returning the replaced SQL and question text
           #     sql = tokenise(sql)
            #    question = preprocess_text(question)
                if not split in datasets:
                    datasets[split] = []
                example = {}
                example["text"] = making_prompt(question)+sql
                example["question"] = question
                example["sql"] = sql
                datasets[split].append(example)
    return datasets

making_prompt = lambda x:x                                    # define prompt constructor as identity
with open("geography.json", 'r') as file:
    geography_data = json.load(file)
    geography_datasets =  build_question_split(geography_data,making_prompt=making_prompt)




"""
original text：
text = "what is the population of CITY0"
variables = {"CITY0": "boston"}
sql = "SELECT population FROM CITY WHERE name = CITY0"


after running：
question = "what is the population of boston"
sql = "SELECT population FROM CITY WHERE name = boston"
text = question + sql
"""

'\noriginal text：\ntext = "what is the population of CITY0"\nvariables = {"CITY0": "boston"}\nsql = "SELECT population FROM CITY WHERE name = CITY0"\n\n\nafter running：\nquestion = "what is the population of boston"\nsql = "SELECT population FROM CITY WHERE name = boston"\ntext = question + sql\n'

In [5]:
#cell 4
import sqlite3

def load_sqlite_file(file_path):
    """
    Load a .sqlite file and return a connection object.

    Args:
        file_path (str): Path to the .sqlite file

    Returns:
        sqlite3.Connection: Connection object to the loaded database
    """
    try:
        conn = sqlite3.connect(file_path)
        print(f"Loaded database from {file_path}")
        return conn
    except sqlite3.Error as e:
        print(f"Error loading database: {e}")
        return None




def get_all_results(dataset, cursor):
    skipped = 0
    total = len(dataset)
    for i, example in enumerate(dataset):
        question = example["question"]
        sql = example["sql"]
        try:
            cursor.execute(sql)                      # send gold SQL to SQLite for execution
            gold_answers = cursor.fetchall()         # take out the query results


        except sqlite3.Error as e:                   # if SQL execution reports an error
            print(f"\n[WARN] Failed to execute gold SQL at index {i}:")
            print(f"  Question: {question}")
            print(f"  SQL: {sql}")
            print(f"  Error: {e}")
            gold_answers = []
            skipped += 1

        example["answers"] = gold_answers

    print(f"\nFinished get_all_results. "
          f"Total: {total}, skipped (error) queries: {skipped}")




def compare_results(generated, answers):
    """
    Compare generated results with gold answers.

    Args:
        generated: List of tuples from executing generated SQL
        answers: List of tuples from executing gold SQL

    Returns:
        tp: True positives (items in both generated and answers)
        fp: False positives (items in generated but not in answers)
        fn: False negatives (items in answers but not in generated)
        exact_match: Boolean indicating if results match exactly
    """
    # Convert to sets for comparison
    generated_set = set(generated) if generated else set()
    answers_set = set(answers) if answers else set()

    # Calculate TP, FP, FN
    tp = len(generated_set & answers_set)  # Intersection
    fp = len(generated_set - answers_set)  # In generated but not in answers
    fn = len(answers_set - generated_set)  # In answers but not in generated

    # Exact match: both sets are identical
    exact_match = (generated_set == answers_set)

    return tp, fp, fn, exact_match



"""
gold answers：[('texas',), ('utah',)]
generated results：[('texas',), ('california',)]

taking into sets：
answers_set = {('texas',), ('utah',)}
generated_set = {('texas',), ('california',)}

and：
tp = 1（texas）
fp = 1（california）
fn = 1（utah）
exact_match = False

"""



def calculate_metrics(all_tp, all_fp, all_fn, exact_matches, total, grammatical_count):
    """
    Calculate micro/macro precision, recall, F1, exact match ratio and grammatical ratio.

    Args:
        all_tp: List of true positives for each example
        all_fp: List of false positives for each example
        all_fn: List of false negatives for each example
        exact_matches: List of exact match booleans
        total: Total number of examples
        grammatical_count: Number of grammatically correct SQL queries

    Returns:
        Dictionary with all metrics
    """
    # Micro metrics (aggregate counts then compute)
    total_tp = sum(all_tp)
    total_fp = sum(all_fp)
    total_fn = sum(all_fn)

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0    # micro precision = TP / (TP+FP)
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0     # micro recall = TP / (TP+FN)
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    # micro F1 = 2PR/(P+R)


    # Macro metrics (compute per example then average)
    precisions = []
    recalls = []
    f1s = []

    for tp, fp, fn in zip(all_tp, all_fp, all_fn):
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)                                     # each example

    macro_precision = sum(precisions) / len(precisions) if precisions else 0.0
    macro_recall = sum(recalls) / len(recalls) if recalls else 0.0
    macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0

    # Exact match and grammatical ratios
    exact_match_ratio = sum(exact_matches) / total if total > 0 else 0.0
    grammatical_ratio = grammatical_count / total if total > 0 else 0.0

    return {
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'exact_match_ratio': exact_match_ratio,
        'grammatical_ratio': grammatical_ratio
    }

print("Evaluation functions defined successfully!")
print(f"Dataset splits: {list(geography_datasets.keys())}")
print(f"Train examples: {len(geography_datasets['train'])}")
print(f"Dev examples: {len(geography_datasets['dev'])}")
print(f"Test examples: {len(geography_datasets['test'])}")


Evaluation functions defined successfully!
Dataset splits: ['dev', 'test', 'train']
Train examples: 549
Dev examples: 49
Test examples: 279


In [6]:
# cell 5
import torch
import re
from tqdm import tqdm
import sqlite3

def extract_sql_from_generation(generated_text, prompt):
    """
    Extract SQL query from generated text.
    Removes the prompt and extracts the SQL part.
    """
    # Remove the prompt from the beginning
    if prompt in generated_text:
        sql_part = generated_text[len(prompt):].strip()         # cut out the prompt at the beginning
    else:
        sql_part = generated_text.strip()                       # remove leading and trailing spaces/newlines


    # Try to extract SQL (ends at semicolon)
    match = re.search(r'(SELECT\s+.*?;)', sql_part, re.IGNORECASE | re.DOTALL)
    # Matches the "SELECT",
    # s+: Matches at least 1 blank character,
    # *? : Matches any character (.) any number of times (*), but? Indicates non-greed (as short as possible)


    """
prompt:
What is the population of Boston?


generated:
What is the population of Boston?
Sure! Here is the SQL you need:
SELECT population FROM CITY WHERE name = 'boston';
This query selects the population of Boston from the CITY table.

we take:
SELECT population FROM CITY WHERE name = 'boston';

    """



    if match:
        return match.group(1).strip()


    # If no semicolon, take until newline or end
    lines = sql_part.split('\n')
    for line in lines:
        if line.strip().upper().startswith('SELECT'):
            return line.strip()


    return sql_part.split('\n')[0].strip() if sql_part else ""
    # if neither SELECT can be found ...; , and no line starting with SELECT can be found: return the first line as "guessed SQL"




def evaluate(dataset, model, conn, tokenizer, making_prompt=lambda x: x,
             grammar_processor=None, max_new_tokens=256, verbose=True):
    """
    Evaluate model on text-to-SQL task.

    Args:
        dataset: List of examples with 'question', 'sql', and optionally 'answers' fields
        model: The language model
        conn: SQLite database connection
        tokenizer: Model tokenizer
        making_prompt: Function to create prompt from question
        grammar_processor: xgrammar LogitsProcessor for constrained generation (optional)
        max_new_tokens: Maximum number of tokens to generate
        verbose: Whether to print progress

    Returns:
        Dictionary with evaluation metrics
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()                # evaluation mode

    cursor = conn.cursor()      # take the database cursor

    all_tp = []
    all_fp = []
    all_fn = []
    exact_matches = []
    grammatical_count = 0       # use for getting tp, fp, fn and exact_match: True/False
    total = len(dataset)

    results = []  # Store detailed results for analysis



    iterator = tqdm(dataset, desc="Evaluating") if verbose else dataset


    with torch.no_grad():
        for example in iterator:
            question = example["question"]             # take out the natural language problem
            gold_sql = example["sql"]                  # take out the standard answer SQL

            q_norm = question.lower().strip()
            if q_norm in [
                "what state borders the most states",
                "which state borders the most states",
            ]:

                gold_sql = """
                SELECT STATE_NAME
                FROM BORDER_INFO
                GROUP BY STATE_NAME
                ORDER BY COUNT(DISTINCT BORDER) DESC
                LIMIT 1;
                """.strip()
            # if the samples are not pre-stored with answers, execute gold SQL on site to get the standard results
            gold_answers = example.get("answers", None)
            if gold_answers is None:
                try:
                    cursor.execute(gold_sql)
                    gold_answers = cursor.fetchall()     # execute gold SQL and get the standard result with fetchall()

                # if fail to execute, give the warning and set the gold answers to empty list
                except sqlite3.Error as e:
                    print(f"[WARN] Failed to execute gold SQL for question:\n  {question}")
                    print(f"  SQL: {gold_sql}")
                    print(f"  Error: {e}")
                    gold_answers = []
                example["answers"] = gold_answers

            # making prompt: give the questions
            prompt = making_prompt(question)

            # Tokenize and generate, turning prompt into tensor: input_ids and attention_mask
            inputs = tokenizer(prompt, return_tensors='pt').to(device)

            generate_kwargs = {
                'max_new_tokens': max_new_tokens,
                'do_sample': False,  # Greedy decoding for reproducibility
                'pad_token_id': tokenizer.eos_token_id,
                'eos_token_id': tokenizer.eos_token_id,
            }

            # Add grammar processor if provided
            if grammar_processor is not None:    # logits_processor will process logits at each generated step
                generate_kwargs['logits_processor'] = [grammar_processor]

            output = model.generate(**inputs, **generate_kwargs)

            # Decode: output.shape == (batch_size, seq_len)
            generated_text = tokenizer.decode(output[0], skip_special_tokens=True)   #?
            generated_sql = extract_sql_from_generation(generated_text, prompt)

            # Try to execute the generated SQL
            is_grammatical = False
            generated_results = []
            error_msg = None

            try:                              # success：take the result, label as is_grammatical=True
                cursor.execute(generated_sql)
                generated_results = cursor.fetchall()
                is_grammatical = True
                grammatical_count += 1
            except sqlite3.Error as e:       # fail: restore the wrong information string
                error_msg = str(e)           # keep it as blank
                generated_results = []

            # Compare results
            tp, fp, fn, exact_match = compare_results(generated_results, gold_answers)

            all_tp.append(tp)
            all_fp.append(fp)
            all_fn.append(fn)
            exact_matches.append(exact_match)

            # Store result for analysis
            results.append({
                'question': question,
                'gold_sql': gold_sql,
                'generated_sql': generated_sql,
                'is_grammatical': is_grammatical,
                'exact_match': exact_match,
                'error': error_msg,
                'tp': tp, 'fp': fp, 'fn': fn
            })

    # Calculate metrics
    metrics = calculate_metrics(all_tp, all_fp, all_fn, exact_matches, total, grammatical_count)
    metrics['detailed_results'] = results

    return metrics


def print_evaluation_results(metrics, name=""):
    """Pretty print evaluation results."""
    print(f"\n{'='*50}")
    print(f"Evaluation Results {name}")
    print(f"{'='*50}")
    print(f"Micro Precision: {metrics['micro_precision']:.4f}")
    print(f"Micro Recall:    {metrics['micro_recall']:.4f}")
    print(f"Micro F1:        {metrics['micro_f1']:.4f}")
    print(f"{'--'*25}")
    print(f"Macro Precision: {metrics['macro_precision']:.4f}")
    print(f"Macro Recall:    {metrics['macro_recall']:.4f}")
    print(f"Macro F1:        {metrics['macro_f1']:.4f}")
    print(f"{'--'*25}")
    print(f"Exact Match:     {metrics['exact_match_ratio']:.4f}")
    print(f"Grammatical:     {metrics['grammatical_ratio']:.4f}")
    print(f"{'='*50}\n")


def analyze_errors(metrics, n=5):
    """Analyze and print error cases."""
    results = metrics.get('detailed_results', [])

    # Grammar errors
    grammar_errors = [r for r in results if not r['is_grammatical']]
    print(f"\n--- Grammar Errors ({len(grammar_errors)} total) ---")
    for r in grammar_errors[:n]:
        print(f"Q: {r['question']}")
        print(f"Generated: {r['generated_sql']}")
        print(f"Error: {r['error']}")
        print()

    # Semantic errors (grammatical but wrong results)
    semantic_errors = [r for r in results if r['is_grammatical'] and not r['exact_match']]
    print(f"\n--- Semantic Errors ({len(semantic_errors)} total) ---")
    for r in semantic_errors[:n]:
        print(f"Q: {r['question']}")
        print(f"Gold: {r['gold_sql']}")
        print(f"Generated: {r['generated_sql']}")
        print()

print("Evaluate function defined with xgrammar support!")


Evaluate function defined with xgrammar support!


In [7]:
#this is just for reference, we need a sql version
!wget https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/geo_query.ebnf
#the json grammar is informative to learn how to write the basic elements e.g., strings numbers
!wget https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/json_minimal.ebnf


--2025-12-18 06:37:34--  https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/geo_query.ebnf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘geo_query.ebnf’

geo_query.ebnf          [ <=>                ] 215.31K  --.-KB/s    in 0.007s  

2025-12-18 06:37:34 (29.2 MB/s) - ‘geo_query.ebnf’ saved [220473]

--2025-12-18 06:37:35--  https://github.com/Saibo-creator/transformers-CFG/blob/main/examples/grammars/json_minimal.ebnf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘json_minimal.ebnf’

json_minimal.ebnf       [ <=>                ] 187.83K  --.-KB/s    in 0.009s  

2025-12-18 06:37:35 (20.4 MB/s) - ‘json_minimal.ebnf’ saved [1923

In [8]:
#cell 7
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import json

# =============================================================================
# Phase 1: Basic Prompting
# =============================================================================

# Database Schema for GeoQuery
DATABASE_SCHEMA = """Database Schema:
- state(state_name, population, area, country_name, capital, density)
- city(city_name, population, country_name, state_name)
- river(river_name, length, country_name, traverse)
- border_info(state_name, border)
- highlow(state_name, highest_elevation, lowest_point, highest_point, lowest_elevation)
- mountain(mountain_name, mountain_altitude, country_name, state_name)
- lake(lake_name, area, country_name, state_name)"""


# prevent it from compiling table names/column names in disorder
# the big model itself doesn't know what database looks like

# Few-shot examples for prompting
FEW_SHOT_EXAMPLES = """
Example 1:
Question: What is the capital of Texas?
SQL: SELECT capital FROM state WHERE state_name = 'texas';

Example 2:
Question: What is the population of New York City?
SQL: SELECT population FROM city WHERE city_name = 'new york';

Example 3:
Question: What rivers run through Colorado?
SQL: SELECT river_name FROM river WHERE traverse = 'colorado';

Example 4:
Question: What is the longest river in the USA?
SQL: SELECT river_name FROM river WHERE length = (SELECT MAX(length) FROM river);

Example 5:
Question: Which states border Texas?
SQL: SELECT border FROM border_info WHERE state_name = 'texas';
"""

#giving LLM examples to prevent it compiling
# ?
def making_prompt_fewshot(question):
    """Create a few-shot prompt with database schema."""
    return f"""You are a SQL expert. Convert natural language questions to SQL queries for a US geography database.

{DATABASE_SCHEMA}

{FEW_SHOT_EXAMPLES}
Now convert this question to SQL:
Question: {question}
SQL: """

def making_prompt_zeroshot(question):
    """Create a zero-shot prompt with database schema (for training and evaluation)."""
    return f"""You are a SQL expert. Convert the following question to a SQL query for the given database.
Return ONLY the SQL query.

{DATABASE_SCHEMA}

Question: {question}
SQL: """


def making_prompt_simple(question):
    """Simple prompt without schema."""
    return f"""Convert to SQL: {question}
SQL: """


def strip_answers(split_data):
    cleaned = []
    for ex in split_data:
        cleaned.append({
            "text": ex.get("text", ""),
            "question": ex.get("question", ""),
            "sql": ex.get("sql", ""),
        })
    return cleaned

# Drop the answers

# Prepare datasets (for HF), turn Python list into HuggingFace Dataset
train_data = Dataset.from_list(strip_answers(geography_datasets["train"]))
dev_data   = Dataset.from_list(strip_answers(geography_datasets["dev"]))
test_data  = Dataset.from_list(strip_answers(geography_datasets["test"]))

dataset = DatasetDict({"train": train_data, "dev": dev_data, "test": test_data})
print(f"Dataset loaded:")
print(f"  Train: {len(dataset['train'])} examples")
print(f"  Dev: {len(dataset['dev'])} examples")
print(f"  Test: {len(dataset['test'])} examples")
print(f"\nSample training example:")
print(f"  Question: {dataset['train'][0]['question']}")
print(f"  SQL: {dataset['train'][0]['sql']}")

# Load the base model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
print(f"\nLoading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token       #some models do not have pad token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")


Dataset loaded:
  Train: 549 examples
  Dev: 49 examples
  Test: 279 examples

Sample training example:
  Question: what is the biggest city in nebraska
  SQL: SELECT CITYalias0.CITY_NAME FROM CITY AS CITYalias0 WHERE CITYalias0.POPULATION = ( SELECT MAX( CITYalias1.POPULATION ) FROM CITY AS CITYalias1 WHERE CITYalias1.STATE_NAME = "nebraska" ) AND CITYalias0.STATE_NAME = "nebraska" ;

Loading model: HuggingFaceTB/SmolLM2-360M-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 361,821,120


In [9]:
#cell 8
# =============================================================================
# Phase 1: Evaluate Baseline Model with Few-shot Prompting
# =============================================================================

# Open database connection for evaluation
conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

# test the prompt format first, print the first 500 characters of prompt
# take the first question from the dev set as an example
test_question = geography_datasets["dev"][0]["question"]
print("Sample prompt (few-shot):")
print("-" * 50)
print(making_prompt_fewshot(test_question)[:500] + "...")
print("-" * 50)

# Evaluate on dev set (smaller for quick testing)
print("\n[Phase 1] Evaluating baseline model with few-shot prompting on DEV set...")
baseline_metrics = evaluate(
    geography_datasets["dev"],
    model,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    grammar_processor=None,
    max_new_tokens=128,
    verbose=True
)


print_evaluation_results(baseline_metrics, name="[Baseline - Few-shot]")

# Analyze some errors
print("\nError Analysis:")
analyze_errors(baseline_metrics, n=3)

conn.close()


Loaded database from geography-db.added-in-2020.sqlite
Sample prompt (few-shot):
--------------------------------------------------
You are a SQL expert. Convert natural language questions to SQL queries for a US geography database.

Database Schema:
- state(state_name, population, area, country_name, capital, density)
- city(city_name, population, country_name, state_name)
- river(river_name, length, country_name, traverse)
- border_info(state_name, border)
- highlow(state_name, highest_elevation, lowest_point, highest_point, lowest_elevation)
- mountain(mountain_name, mountain_altitude, country_name, state_name)
- lake(lak...
--------------------------------------------------

[Phase 1] Evaluating baseline model with few-shot prompting on DEV set...


Evaluating:  92%|█████████▏| 45/49 [02:03<00:11,  2.96s/it]

[WARN] Failed to execute gold SQL for question:
  which state borders most states
  SQL: SELECT DERIVED_TABLEalias1.STATE_NAME FROM ( SELECT BORDER_INFOalias0.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias0.BORDER ) AS DERIVED_FIELDalias0 FROM BORDER_INFO AS BORDER_INFOalias0 GROUP BY BORDER_INFOalias0.STATE_NAME ) AS DERIVED_TABLEalias0 WHERE DERIVED_TABLEalias0.DERIVED_FIELDalias0 = ( SELECT MAX( DERIVED_TABLEalias1.DERIVED_FIELDalias1 ) FROM ( SELECT BORDER_INFOalias1.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias1.BORDER ) AS DERIVED_FIELDalias1 FROM BORDER_INFO AS BORDER_INFOalias1 GROUP BY BORDER_INFOalias1.STATE_NAME ) AS DERIVED_TABLEalias1 ) ;
  Error: no such column: DERIVED_TABLEalias1.STATE_NAME


Evaluating: 100%|██████████| 49/49 [02:08<00:00,  2.61s/it]


Evaluation Results [Baseline - Few-shot]
Micro Precision: 0.1812
Micro Recall:    0.1453
Micro F1:        0.1613
--------------------------------------------------
Macro Precision: 0.3681
Macro Recall:    0.3524
Macro F1:        0.3376
--------------------------------------------------
Exact Match:     0.3469
Grammatical:     0.7959


Error Analysis:

--- Grammar Errors (10 total) ---
Q: what is the biggest city in arizona
Generated: SELECT city FROM city WHERE city_name = 'albuquerque';
Error: no such column: city

Q: what texas city has the largest population
Generated: SELECT city FROM city WHERE city_name = 'texas';
Error: no such column: city

Q: what is the largest city in missouri
Generated: SELECT city FROM city WHERE city_name = 'jacksonville';
Error: no such column: city


--- Semantic Errors (22 total) ---
Q: how many people live in washington
Gold: SELECT STATEalias0.POPULATION FROM STATE AS STATEalias0 WHERE STATEalias0.STATE_NAME = "washington" ;
Generated: SELECT count(

In [10]:
# cell 9
# =============================================================================
# Phase 2: Fine-tuning with LoRA + SFTTrainer (Zero-shot + Completion-only Loss)
# =============================================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model
# ---- Custom completion-only collator (works even if TRL lacks DataCollatorForCompletionOnlyLM) ----
from dataclasses import dataclass
from typing import Any, Dict, List

def _find_sublist(haystack: List[int], needle: List[int]) -> int:
    """Return the first index where needle occurs in haystack, or -1 if not found."""
    n = len(needle)
    if n == 0:
        return -1
    for i in range(len(haystack) - n + 1):
        if haystack[i:i+n] == needle:
            return i
    return -1

@dataclass
class CompletionOnlyCollator:
    """Mask loss on the prompt and compute loss only on the completion after a response template."""
    tokenizer: Any
    response_template: str = "SQL:"

    def __post_init__(self):
        self.response_ids = self.tokenizer(self.response_template, add_special_tokens=False).input_ids
        if not self.response_ids:
            raise ValueError("response_template tokenized to empty ids. Check your template string.")

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # SFTTrainer typically gives tokenized features with input_ids/attention_mask
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        labels = input_ids.clone()

        for i in range(input_ids.size(0)):
            seq = input_ids[i].tolist()
            pos = _find_sublist(seq, self.response_ids)
            if pos == -1:
                # If template not found, ignore this sample to avoid training on wrong spans
                labels[i].fill_(-100)
                continue
            start = pos + len(self.response_ids)
            labels[i, :start] = -100  # mask everything up to and including "SQL:"

        # Also mask padding tokens
        labels[attention_mask == 0] = -100
        batch["labels"] = labels
        return batch

print("Loading fresh model for fine-tuning...")

# Reload a new base model for fine tuning, avoid polluting baseline
model_for_finetuning = AutoModelForCausalLM.from_pretrained(model_name)
model_for_finetuning.config.pad_token_id = tokenizer.pad_token_id

# ---------------- LoRA deploy ---------------- Low-Rank Adaptation
lora_config = LoraConfig(
    r=64,                      # LoRA rank
    lora_alpha=128,             # LoRA scaling, control the update range of LoRA
    lora_dropout=0.1,          # LoRA dropout, anti overfitting
    bias="none",
    task_type="CAUSAL_LM",     # tell PEFT that this is an autoregressive language model task
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # must match module names in the model
)

# Using LoRA
model_with_lora = get_peft_model(model_for_finetuning, lora_config)
# Insert the LoRA low-rank adaptation layer into the specified modules

model_with_lora.print_trainable_parameters()
# Print trainable parameters and proportion

# ---------------- constructing training data ----------------
def format_training_example(example):
    """Format example for supervised fine-tuning (ZERO-shot prompt)."""
    prompt = making_prompt_zeroshot(example["question"])  # Use zero-shot prompt
    # IMPORTANT: The prompt must end with "SQL:" so the completion-only collator can mask correctly
    return {"text": prompt + example["sql"] + tokenizer.eos_token}

train_formatted = dataset["train"].map(format_training_example)
dev_formatted   = dataset["dev"].map(format_training_example)

print("\nFormatted training example:")
print(train_formatted[0]["text"][:300] + "...")

# ---------------- Sanity check: "SQL:" template must exist ----------------
sample = train_formatted[0]["text"]
ids = tokenizer(sample, add_special_tokens=False).input_ids
tpl = tokenizer("SQL:", add_special_tokens=False).input_ids
print("template ids:", tpl)
print("template found:", _find_sublist(ids, tpl) != -1)


# ---------------- SFTConfig（replacing TrainingArguments）----------------
sft_config = SFTConfig(
    output_dir="./smollm2-sql-lora",      # saving directory
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,

    # SFTConfig uses eval_strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    report_to="none",

    # Dataset field that contains the full prompt+answer text
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

# ---------------- Completion-only data collator ----------------
# This masks out loss on the prompt and ONLY computes loss on the completion after "SQL:"
collator = CompletionOnlyCollator(tokenizer=tokenizer, response_template="SQL:")

# ---------------- Making SFTTrainer ----------------
trainer = SFTTrainer(
    model=model_with_lora,       # LoRA model
    args=sft_config,             # SFTConfig
    train_dataset=train_formatted,
    eval_dataset=dev_formatted,
    processing_class=tokenizer,  # tokenizer
    data_collator=collator,      # completion-only loss
)

print("\nStarting fine-tuning...")
print(f"Training samples: {len(train_formatted)}")
print(f"Evaluation samples: {len(dev_formatted)}")


Loading fresh model for fine-tuning...
trainable params: 13,107,200 || all params: 374,928,320 || trainable%: 3.4959


Map:   0%|          | 0/549 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]


Formatted training example:
You are a SQL expert. Convert the following question to a SQL query for the given database.
Return ONLY the SQL query.

Database Schema:
- state(state_name, population, area, country_name, capital, density)
- city(city_name, population, country_name, state_name)
- river(river_name, length, country_n...
template ids: [15933, 42]
template found: True


Adding EOS to train dataset:   0%|          | 0/549 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/549 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/49 [00:00<?, ? examples/s]


Starting fine-tuning...
Training samples: 549
Evaluation samples: 49


In [11]:
#cell 10
# =============================================================================
# Run Training
# =============================================================================

# Train the model
trainer.train()

# Save the fine-tuned model
trainer.save_model()
print("\nFine-tuned model saved to ./smollm2-sql-lora")

# You can also save just the LoRA weights
model_with_lora.save_pretrained("./smollm2-sql-lora-adapter")
print("LoRA adapter saved to ./smollm2-sql-lora-adapter")


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.270100,0.171451,1.204630,130197.000000,0.949941
2,0.109300,0.110287,1.118837,260394.000000,0.966735
3,0.076100,0.100841,1.094945,390591.000000,0.972327



Fine-tuned model saved to ./smollm2-sql-lora
LoRA adapter saved to ./smollm2-sql-lora-adapter


In [12]:
#cell 11
# =============================================================================
# Evaluate Fine-tuned Model
# =============================================================================

# Open database connection
conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

# Evaluate fine-tuned model on dev set
print("\n[Phase 2] Evaluating FINE-TUNED model on DEV set...")
finetuned_metrics = evaluate(
    geography_datasets["dev"],                     # testing datasets dev
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_zeroshot,
    grammar_processor=None,
    max_new_tokens=256,
    verbose=True
)

print_evaluation_results(finetuned_metrics, name="[Fine-tuned - Few-shot]")

# Compare with baseline
print("\n" + "="*50)
print("COMPARISON: Baseline vs Fine-tuned")
print("="*50)
print(f"{'Metric':<20} {'Baseline':<12} {'Fine-tuned':<12} {'Δ':<10}")
print("-"*50)
for metric in ['exact_match_ratio', 'grammatical_ratio', 'micro_f1', 'macro_f1']:
    baseline_val = baseline_metrics[metric]
    finetuned_val = finetuned_metrics[metric]
    delta = finetuned_val - baseline_val
    print(f"{metric:<20} {baseline_val:<12.4f} {finetuned_val:<12.4f} {delta:+.4f}")

# Error analysis
print("Fine-tuned Model Error Analysis:")
analyze_errors(finetuned_metrics, n=3)

conn.close()


Loaded database from geography-db.added-in-2020.sqlite

[Phase 2] Evaluating FINE-TUNED model on DEV set...


Evaluating: 100%|██████████| 49/49 [03:04<00:00,  3.77s/it]


Evaluation Results [Fine-tuned - Few-shot]
Micro Precision: 0.4762
Micro Recall:    0.3488
Micro F1:        0.4027
--------------------------------------------------
Macro Precision: 0.6488
Macro Recall:    0.6592
Macro F1:        0.6429
--------------------------------------------------
Exact Match:     0.6531
Grammatical:     0.9388


COMPARISON: Baseline vs Fine-tuned
Metric               Baseline     Fine-tuned   Δ         
--------------------------------------------------
exact_match_ratio    0.3469       0.6531       +0.3061
grammatical_ratio    0.7959       0.9388       +0.1429
micro_f1             0.1613       0.4027       +0.2414
macro_f1             0.3376       0.6429       +0.3053
Fine-tuned Model Error Analysis:

--- Grammar Errors (3 total) ---
Q: which rivers run through the state with the largest city in the us
Generated: SELECT RIVERalias0.RIVER_NAME FROM RIVER AS RIVERalias0 WHERE RIVERalias0.TRAVERSE = "us" AND RIVERalias0.LENGTH = ( SELECT MAX( RIVERalias1.LENGTH 

In [ ]:
#cell 12
# =============================================================================
# Phase 3: Context-Free Grammar (CFG) Constrained Generation with xgrammar
# =============================================================================

import xgrammar as xgr

# Load the SQL grammar from EBNF file
sql_grammar_ebnf = """
root ::= select_stmt

select_stmt ::= "SELECT " select_list " FROM " table_name where_clause? ";"

select_list ::= select_item (", " select_item)*
select_item ::= "*" | column_name | aggregate_func

aggregate_func ::= agg_name "(" column_name_or_star ")"
agg_name ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"
column_name_or_star ::= column_name | "*"

column_name ::= [a-z_]+
table_name ::= [a-z_]+

where_clause ::= " WHERE " condition

condition ::= simple_condition (" AND " simple_condition)* | simple_condition (" OR " simple_condition)*

simple_condition ::= column_name " " comparison_op " " value
                   | column_name " IN (" subquery ")"
                   | column_name " " comparison_op " (" subquery ")"

comparison_op ::= "=" | "!=" | "<" | ">" | "<=" | ">="

value ::= string_literal | number | column_name
string_literal ::= "'" [a-z0-9 _-]* "'"
number ::= [0-9]+

subquery ::= select_stmt
"""

# Create xgrammar components
print("Setting up xgrammar for constrained generation...")

# Get tokenizer info from HuggingFace tokenizer, let xgrammar know tokenizer
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=tokenizer.vocab_size)
# Convert the tokenizer information of HF into TokenizerInfo, a format that xgrammar can use


# Compile the grammar
grammar_compiler = xgr.GrammarCompiler(tokenizer_info)    # create a compiler
compiled_grammar = grammar_compiler.compile_grammar(sql_grammar_ebnf)
# Compile the written EBNF grammar into a constraint object that can be used when generating

print("Grammar compiled successfully!")
print(f"Grammar EBNF preview:\n{sql_grammar_ebnf[:500]}...")

Setting up xgrammar for constrained generation...
Grammar compiled successfully!
Grammar EBNF preview:

root ::= select_stmt

select_stmt ::= "SELECT " select_list " FROM " table_name where_clause? ";"

select_list ::= select_item (", " select_item)*
select_item ::= "*" | column_name | aggregate_func

aggregate_func ::= agg_name "(" column_name_or_star ")"
agg_name ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"
column_name_or_star ::= column_name | "*"

column_name ::= [a-z_]+
table_name ::= [a-z_]+

where_clause ::= " WHERE " condition

condition ::= simple_condition (" AND " simple_condition)* | s...


In [ ]:
#cell 13
# =============================================================================
# Evaluate with CFG-Constrained Generation
# =============================================================================

def evaluate_with_cfg(dataset, model, conn, tokenizer, making_prompt,
                      compiled_grammar, tokenizer_info, max_new_tokens=128, verbose=True):
    """
    Evaluate model with xgrammar CFG-constrained generation.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()                 # evaluation mode

    cursor = conn.cursor()       # connect to the cursor

    all_tp = []
    all_fp = []
    all_fn = []
    exact_matches = []
    grammatical_count = 0
    total = len(dataset)        # use for getting tp, fp, fn and exact_match: True/False

    results = []

    iterator = tqdm(dataset, desc="Evaluating with CFG") if verbose else dataset

    with torch.no_grad():
        for example in iterator:
            question = example["question"]           # take out the natural language problem
            gold_sql = example["sql"]                # take out the standard answer SQL

            # if the samples are not pre-stored with answers, execute gold SQL on site to get the standard results
            gold_answers = example.get("answers", None)
            if gold_answers is None:
                try:
                    cursor.execute(gold_sql)
                    gold_answers = cursor.fetchall()
                except sqlite3.Error as e:
                    print(f"[WARN] Failed to execute gold SQL for question:\n  {question}")
                    print(f"  SQL: {gold_sql}")
                    print(f"  Error: {e}")
                    gold_answers = []
            # --------------------------------------
                example["answers"] = gold_answers

            # making prompt: give the questions
            prompt = making_prompt(question)

            # Tokenize and generate, turning prompt into tensor: input_ids and attention_mask
            inputs = tokenizer(prompt, return_tensors='pt').to(device)


            # Create a new grammar matcher for each generation
            # Package compiled_grammar into LogitsProcessor that HuggingFace generate can use
            # Every time a token is generated, the probability of a token that does not conform to grammar is cut off/masked
            xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

            # Generate with CFG constraints
            try:
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    logits_processor=[xgr_logits_processor],  # taking CFG into
                )

                generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
                # Return the token ids output by the model to a string
                generated_sql = extract_sql_from_generation(generated_text, prompt)
                # Extract SQL from generated text
            except Exception as e:
                # If CFG generation fails, fall back to empty SQL
                generated_sql = ""

            # Try to execute the generated SQL
            is_grammatical = False
            generated_results = []
            error_msg = None

            try:
                cursor.execute(generated_sql)
                generated_results = cursor.fetchall()
                is_grammatical = True
                grammatical_count += 1
            except sqlite3.Error as e:
                error_msg = str(e)
                generated_results = []


            # Compare results
            tp, fp, fn, exact_match = compare_results(generated_results, gold_answers)

            all_tp.append(tp)
            all_fp.append(fp)
            all_fn.append(fn)
            exact_matches.append(exact_match)

            results.append({
                'question': question,
                'gold_sql': gold_sql,
                'generated_sql': generated_sql,
                'is_grammatical': is_grammatical,
                'exact_match': exact_match,
                'error': error_msg,
                'tp': tp, 'fp': fp, 'fn': fn
            })

    metrics = calculate_metrics(all_tp, all_fp, all_fn, exact_matches, total, grammatical_count)
    metrics['detailed_results'] = results

    return metrics


# Evaluate fine-tuned model with CFG constraints
conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

print("[Phase 3] Evaluating FINE-TUNED model WITH CFG constraints on DEV set...")
cfg_metrics = evaluate_with_cfg(
    geography_datasets["dev"],
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    compiled_grammar=compiled_grammar,
    tokenizer_info=tokenizer_info,
    max_new_tokens=128,
    verbose=True
)

print_evaluation_results(cfg_metrics, name="[Fine-tuned + CFG]")

# Error analysis for CFG
print("CFG Constrained Model Error Analysis:")
analyze_errors(cfg_metrics, n=3)

conn.close()


Loaded database from geography-db.added-in-2020.sqlite
[Phase 3] Evaluating FINE-TUNED model WITH CFG constraints on DEV set...


Evaluating with CFG: 100%|██████████| 49/49 [04:08<00:00,  5.06s/it]


Evaluation Results [Fine-tuned + CFG]
Micro Precision: 0.3636
Micro Recall:    0.0698
Micro F1:        0.1171
--------------------------------------------------
Macro Precision: 0.0857
Macro Recall:    0.0898
Macro F1:        0.0871
--------------------------------------------------
Exact Match:     0.1020
Grammatical:     0.2245

CFG Constrained Model Error Analysis:

--- Grammar Errors (38 total) ---
Q: what is the biggest city in arizona
Generated: SELECT COUNT(city) FROM city WHERE city_name = 'arizona' OR city_name = (SELECT MAX(city_name) FROM city WHERE city_name != (SELECT MIN(city_name) FROM city WHERE state_name = 'arizona' OR state_name = 'baja' OR state_name = 'jules' OR state_name = 'jules_baja' OR state_name = 'jules_baja_1' OR state_name = 'jules_baja_2' OR state_name = 'jules_baja_3
Error: unrecognized token: "'jules_baja_3"

Q: what texas city has the largest population
Generated: SELECT COUNT(city) FROM state WHERE state_name = 'texas' AND capital = 'new york' AND st

In [ ]:
#cell 14
# =============================================================================
# Phase 4: Additional Experiments & Comprehensive Comparison
# =============================================================================

import pandas as pd

# Test CFG + Prompting WITHOUT Fine-tuning (baseline + CFG)
print("[Phase 4] Additional Experiment: Baseline model WITH CFG constraints...")
conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

# Reload fresh baseline model
baseline_model_fresh = AutoModelForCausalLM.from_pretrained(model_name)
baseline_model_fresh.config.pad_token_id = tokenizer.pad_token_id

baseline_cfg_metrics = evaluate_with_cfg(
    geography_datasets["dev"],
    baseline_model_fresh,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    compiled_grammar=compiled_grammar,
    tokenizer_info=tokenizer_info,
    max_new_tokens=128,
    verbose=True
)

print_evaluation_results(baseline_cfg_metrics, name="[Baseline + CFG (No Fine-tuning)]")

conn.close()

[Phase 4] Additional Experiment: Baseline model WITH CFG constraints...
Loaded database from geography-db.added-in-2020.sqlite


Evaluating with CFG: 100%|██████████| 49/49 [01:25<00:00,  1.74s/it]


Evaluation Results [Baseline + CFG (No Fine-tuning)]
Micro Precision: 0.1562
Micro Recall:    0.1163
Micro F1:        0.1333
--------------------------------------------------
Macro Precision: 0.2461
Macro Recall:    0.2857
Macro F1:        0.2472
--------------------------------------------------
Exact Match:     0.2653
Grammatical:     0.5918



In [ ]:
#cell 15
# =============================================================================
# Comprehensive Results Summary
# =============================================================================

# Collect all results
all_experiments = {
    'Baseline (Few-shot)': baseline_metrics,
    'Fine-tuned (Few-shot)': finetuned_metrics,
    'Baseline + CFG': baseline_cfg_metrics,
    'Fine-tuned + CFG': cfg_metrics,
}

# Create comparison table
metrics_to_compare = ['exact_match_ratio', 'grammatical_ratio', 'micro_f1', 'macro_f1',
                      'micro_precision', 'micro_recall']

results_data = []
for exp_name, metrics in all_experiments.items():
    row = {'Configuration': exp_name}
    for metric in metrics_to_compare:
        row[metric] = metrics.get(metric, 0)
    results_data.append(row)

results_df = pd.DataFrame(results_data)

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Highlight key findings
print(" KEY FINDINGS:")
print("-"*40)

# Best exact match
best_em_idx = results_df['exact_match_ratio'].idxmax()
print(f"Best Exact Match: {results_df.loc[best_em_idx, 'Configuration']} "
      f"({results_df.loc[best_em_idx, 'exact_match_ratio']:.4f})")

# Best grammatical ratio
best_gram_idx = results_df['grammatical_ratio'].idxmax()
print(f"Best Grammatical: {results_df.loc[best_gram_idx, 'Configuration']} "
      f"({results_df.loc[best_gram_idx, 'grammatical_ratio']:.4f})")

# Best F1
best_f1_idx = results_df['micro_f1'].idxmax()
print(f"Best Micro F1: {results_df.loc[best_f1_idx, 'Configuration']} "
      f"({results_df.loc[best_f1_idx, 'micro_f1']:.4f})")

print("\n" + "="*80)



COMPREHENSIVE RESULTS COMPARISON
        Configuration  exact_match_ratio  grammatical_ratio  micro_f1  macro_f1  micro_precision  micro_recall
  Baseline (Few-shot)           0.346939           0.795918  0.161290  0.337612         0.181159      0.145349
Fine-tuned (Few-shot)           0.265306           0.408163  0.126984  0.244898         0.705882      0.069767
       Baseline + CFG           0.265306           0.591837  0.133333  0.247223         0.156250      0.116279
     Fine-tuned + CFG           0.102041           0.224490  0.117073  0.087075         0.363636      0.069767
 KEY FINDINGS:
----------------------------------------
Best Exact Match: Baseline (Few-shot) (0.3469)
Best Grammatical: Baseline (Few-shot) (0.7959)
Best Micro F1: Baseline (Few-shot) (0.1613)



In [ ]:
#cell 16
# =============================================================================
# Final Evaluation on TEST Set (Best Configuration)
# =============================================================================

# Determine best model based on dev set performance
print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

# Evaluate the fine-tuned + CFG model (typically best) on test set
print("Evaluating BEST model (Fine-tuned + CFG) on TEST set...")
test_metrics_cfg = evaluate_with_cfg(
    geography_datasets["test"],
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    compiled_grammar=compiled_grammar,
    tokenizer_info=tokenizer_info,
    max_new_tokens=128,
    verbose=True
)

print_evaluation_results(test_metrics_cfg, name="[TEST SET - Fine-tuned + CFG]")


# Also evaluate fine-tuned without CFG for comparison
print("\nEvaluating Fine-tuned model (no CFG) on TEST set...")
test_metrics_finetuned = evaluate(
    geography_datasets["test"],
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    grammar_processor=None,
    max_new_tokens=128,
    verbose=True
)

print_evaluation_results(test_metrics_finetuned, name="[TEST SET - Fine-tuned only]")

conn.close()

# Final summary
print("\n" + "="*60)
print("FINAL TEST SET RESULTS SUMMARY")
print("="*60)
print(f"{'Configuration':<25} {'Exact Match':<15} {'Grammatical':<15} {'Micro F1':<15}")
print("-"*60)
print(f"{'Fine-tuned + CFG':<25} {test_metrics_cfg['exact_match_ratio']:<15.4f} "
      f"{test_metrics_cfg['grammatical_ratio']:<15.4f} {test_metrics_cfg['micro_f1']:<15.4f}")
print(f"{'Fine-tuned only':<25} {test_metrics_finetuned['exact_match_ratio']:<15.4f} "
      f"{test_metrics_finetuned['grammatical_ratio']:<15.4f} {test_metrics_finetuned['micro_f1']:<15.4f}")
print("="*60)

print("\n Evaluation complete! Results saved for report generation.")



FINAL TEST SET EVALUATION
Loaded database from geography-db.added-in-2020.sqlite
Evaluating BEST model (Fine-tuned + CFG) on TEST set...


Evaluating with CFG:  37%|███▋      | 103/279 [06:48<21:43,  7.41s/it]

[WARN] Failed to execute gold SQL for question:
  what state borders the most states
  SQL: SELECT DERIVED_TABLEalias1.STATE_NAME FROM ( SELECT BORDER_INFOalias0.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias0.BORDER ) AS DERIVED_FIELDalias0 FROM BORDER_INFO AS BORDER_INFOalias0 GROUP BY BORDER_INFOalias0.STATE_NAME ) AS DERIVED_TABLEalias0 WHERE DERIVED_TABLEalias0.DERIVED_FIELDalias0 = ( SELECT MAX( DERIVED_TABLEalias1.DERIVED_FIELDalias1 ) FROM ( SELECT BORDER_INFOalias1.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias1.BORDER ) AS DERIVED_FIELDalias1 FROM BORDER_INFO AS BORDER_INFOalias1 GROUP BY BORDER_INFOalias1.STATE_NAME ) AS DERIVED_TABLEalias1 ) ;
  Error: no such column: DERIVED_TABLEalias1.STATE_NAME


Evaluating with CFG:  37%|███▋      | 104/279 [06:55<21:19,  7.31s/it]

[WARN] Failed to execute gold SQL for question:
  which state borders the most states
  SQL: SELECT DERIVED_TABLEalias1.STATE_NAME FROM ( SELECT BORDER_INFOalias0.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias0.BORDER ) AS DERIVED_FIELDalias0 FROM BORDER_INFO AS BORDER_INFOalias0 GROUP BY BORDER_INFOalias0.STATE_NAME ) AS DERIVED_TABLEalias0 WHERE DERIVED_TABLEalias0.DERIVED_FIELDalias0 = ( SELECT MAX( DERIVED_TABLEalias1.DERIVED_FIELDalias1 ) FROM ( SELECT BORDER_INFOalias1.STATE_NAME , COUNT( DISTINCT BORDER_INFOalias1.BORDER ) AS DERIVED_FIELDalias1 FROM BORDER_INFO AS BORDER_INFOalias1 GROUP BY BORDER_INFOalias1.STATE_NAME ) AS DERIVED_TABLEalias1 ) ;
  Error: no such column: DERIVED_TABLEalias1.STATE_NAME


Evaluating with CFG: 100%|██████████| 279/279 [20:31<00:00,  4.42s/it]



Evaluation Results [TEST SET - Fine-tuned + CFG]
Micro Precision: 0.7447
Micro Recall:    0.0704
Micro F1:        0.1286
--------------------------------------------------
Macro Precision: 0.1362
Macro Recall:    0.1309
Macro F1:        0.1320
--------------------------------------------------
Exact Match:     0.1577
Grammatical:     0.2832


Evaluating Fine-tuned model (no CFG) on TEST set...


Evaluating: 100%|██████████| 279/279 [14:25<00:00,  3.10s/it]


Evaluation Results [TEST SET - Fine-tuned only]
Micro Precision: 0.3026
Micro Recall:    0.1156
Micro F1:        0.1673
--------------------------------------------------
Macro Precision: 0.1983
Macro Recall:    0.2032
Macro F1:        0.1903
--------------------------------------------------
Exact Match:     0.2115
Grammatical:     0.4875


FINAL TEST SET RESULTS SUMMARY
Configuration             Exact Match     Grammatical     Micro F1       
------------------------------------------------------------
Fine-tuned + CFG          0.1577          0.2832          0.1286         
Fine-tuned only           0.2115          0.4875          0.1673         

 Evaluation complete! Results saved for report generation.


In [ ]:
import torch
import xgrammar as xgr

# 确保模型在正确的 device 上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# CFG 的 logits processor（只建一次）
cfg_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

# pad_token_id 兜底一下，避免有些 tokenizer 没设 eos
pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id

# 要测的 (question, gold_sql)
examples = [
    ("what is the capital of florida",
     "SELECT capital FROM state WHERE state_name = 'florida';"),

    ("what is the capital of california",
     "SELECT capital FROM state WHERE state_name = 'california';"),

    ("what is the capital of new york",
     "SELECT capital FROM state WHERE state_name = 'new york';"),

    ("what is the population of los angeles",
     "SELECT population FROM city WHERE city_name = 'los angeles';"),

    ("what is the population of chicago",
     "SELECT population FROM city WHERE city_name = 'chicago';"),

    ("what is the population of houston",
     "SELECT population FROM city WHERE city_name = 'houston';"),

    ("which states border california",
     "SELECT border FROM border_info WHERE state_name = 'california';"),

    ("which states border new mexico",
     "SELECT border FROM border_info WHERE state_name = 'new mexico';"),

    ("which states border florida",
     "SELECT border FROM border_info WHERE state_name = 'florida';"),

    ("what rivers run through texas",
     "SELECT river_name FROM river WHERE traverse = 'texas';"),

    ("what rivers run through utah",
     "SELECT river_name FROM river WHERE traverse = 'utah';"),

    ("what is the longest river in the usa",
     "SELECT river_name FROM river WHERE length = (SELECT MAX(length) FROM river);"),
]

for q, gold in examples:
    # 用你训练/评估时的一模一样的 prompt
    prompt = making_prompt_fewshot(q)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0,
            do_sample=False,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
            logits_processor=[cfg_processor],  # ★ 这里就是“微调 + CFG”
        )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    # 用你自己写的两参版本 extract_sql_from_generation
    pred_sql = extract_sql_from_generation(generated_text, prompt).strip()

    print("=" * 80)
    print("Q     :", q)
    print("GOLD  :", gold)
    print("PRED  :", pred_sql)
    print("MATCH :", pred_sql == gold)

ModuleNotFoundError: No module named 'xgrammar'